# WIN AS MUCH AS YOU CAN – with AI agents, in Google Colab

Five groups of three AI agents play a version of the prisoner's dilemma. Each group confers before handing in X or Y, and before the high-stake rounds each group sends a representative to negotiate with the others.

Run the cells one at a time, from top to bottom (Shift + Enter). Colab forgets everything when the session ends, so download your logs before you close the tab.

## 1. Get the code and install

In [ ]:
import os, glob, subprocess
from google.colab import files

REPO = "https://github.com/YetiKetil/hierarchy_agents"
FOLDER = "/content/hierarchy_agents"
if os.path.exists(FOLDER):
    # the code is already here from earlier in this session: fetch the latest version from GitHub
    subprocess.run(["git", "-C", FOLDER, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, FOLDER], check=True)
os.chdir(FOLDER + "/win_as_much")
!pip install -q -r ../requirements.txt
print("Working in:", os.getcwd())

## 2. Free test: random participants, no API key

In [ ]:
!python wam_run.py conditions/standard.yaml --test

## 3. Add your API key

Click the **key icon** (Secrets) on the left, add `GEMINI_API_KEY` or `ANTHROPIC_API_KEY` with your key (once, on one line), switch on *Notebook access*, and run the cell.

In [ ]:
from google.colab import userdata
for name in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    lines = [l.strip() for l in value.splitlines() if l.strip()]
    if not lines:
        print(name, "not set")
        continue
    os.environ[name] = lines[0]
    print(name, "found")
    if len(lines) > 1:
        print("  Warning: the secret has more than one line. Only the first line is used.")

## 4. A short game: five rounds

Includes the first representatives' meeting (round 5). Choose your model in the first lines. With `--verbose` you see the meeting and the reports as they happen.

In [ ]:
MODEL = "gemini-3.5-flash-lite"      # Gemini
# MODEL = "claude-haiku-5-5"         # Claude

!python wam_run.py conditions/standard.yaml --model {MODEL} --stop-after 5 --verbose --concurrent 2

## 5. The full game

Ten rounds, stopped without warning – as in the classroom. Change `CONDITION` to try another set-up:
- `known_end` – the agents are told the game ends after round 10
- `two_bot_groups` – two rule-based groups: one copies the majority, one always chooses X
- `first_mover` – one group is willing to move first and choose Y without conditions
- `competitive_group` – one group reads the task as finishing ahead of the others
- `disguised` – the same game without its name and familiar wording

Try the same condition with both models: the results can be very different.

In [ ]:
CONDITION = "standard"
!python wam_run.py conditions/{CONDITION}.yaml --model {MODEL} --concurrent 2

## 6. Read the latest game

In [ ]:
latest = max(glob.glob("logs/*/*.json"), key=os.path.getmtime)
print(latest)
!python wam_show.py "{latest}" 2>/dev/null | head -150

## 7. Download your logs and open them in `wam_replay.html` on your own computer

In [ ]:
import shutil
shutil.make_archive("my_wam_logs", "zip", "logs")
files.download("my_wam_logs.zip")